In [1]:
from pymongo import MongoClient
import pandas as pd
from datetime import datetime

client = MongoClient('mongodb://localhost:27017/')

db = client['ecommerce_db']
clientes = db['clientes']
pedidos = db['pedidos']
produtos = db['produtos']

print("✅ Conectado ao MongoDB com sucesso!")
print(f"📦 Banco de dados: ecommerce_db")
print(f"📋 Coleções: clientes, pedidos, produtos")

✅ Conectado ao MongoDB com sucesso!
📦 Banco de dados: ecommerce_db
📋 Coleções: clientes, pedidos, produtos


In [2]:

print("📝 INSERINDO DADOS NO MONGODB")
print("=" * 40)


clientes_data = [
    {
        "nome": "João Casimiro",
        "email": "joao@email.com",
        "idade": 35,
        "cidade": "Araraquara",
        "estado": "SP",
        "data_cadastro": datetime.now()
    },
    {
        "nome": "Maria Silva",
        "email": "maria@email.com",
        "idade": 28,
        "cidade": "São Paulo",
        "estado": "SP",
        "data_cadastro": datetime.now()
    },
    {
        "nome": "Carlos Santos",
        "email": "carlos@email.com",
        "idade": 42,
        "cidade": "Rio de Janeiro",
        "estado": "RJ",
        "data_cadastro": datetime.now()
    },
    {
        "nome": "Ana Oliveira",
        "email": "ana@email.com",
        "idade": 31,
        "cidade": "Belo Horizonte",
        "estado": "MG",
        "data_cadastro": datetime.now()
    },
    {
        "nome": "Pedro Costa",
        "email": "pedro@email.com",
        "idade": 25,
        "cidade": "Curitiba",
        "estado": "PR",
        "data_cadastro": datetime.now()
    }
]

resultado = clientes.insert_many(clientes_data)
print(f"✅ {len(resultado.inserted_ids)} clientes inseridos!")

produtos_data = [
    {"nome": "Notebook", "categoria": "Eletrônicos", "preco": 3500.00, "estoque": 10},
    {"nome": "Mouse Gamer", "categoria": "Eletrônicos", "preco": 150.00, "estoque": 50},
    {"nome": "Tênis Casual", "categoria": "Calçados", "preco": 280.00, "estoque": 30},
    {"nome": "Camiseta", "categoria": "Roupas", "preco": 89.90, "estoque": 100},
    {"nome": "Livro Python", "categoria": "Livros", "preco": 59.90, "estoque": 20}
]

resultado_prod = produtos.insert_many(produtos_data)
print(f"✅ {len(resultado_prod.inserted_ids)} produtos inseridos!")

📝 INSERINDO DADOS NO MONGODB
✅ 5 clientes inseridos!
✅ 5 produtos inseridos!


In [3]:
print("🔍 BUSCANDO DADOS NO MONGODB")
print("=" * 40)

print("\n📋 TODOS OS CLIENTES:")
for cliente in clientes.find():
    print(f"• {cliente['nome']} — {cliente['cidade']}/{cliente['estado']}")

print("\n📍 CLIENTES DE SP:")
for cliente in clientes.find({"estado": "SP"}):
    print(f"• {cliente['nome']} — {cliente['cidade']}")

print("\n👤 BUSCA POR NOME:")
resultado = clientes.find_one({"nome": "João Casimiro"})
print(f"• Nome: {resultado['nome']}")
print(f"• Email: {resultado['email']}")
print(f"• Idade: {resultado['idade']}")

print(f"\n📊 Total de clientes: {clientes.count_documents({})}")
print(f"📊 Clientes de SP: {clientes.count_documents({'estado': 'SP'})}")

🔍 BUSCANDO DADOS NO MONGODB

📋 TODOS OS CLIENTES:
• João Casimiro — Araraquara/SP
• Maria Silva — São Paulo/SP
• Carlos Santos — Rio de Janeiro/RJ
• Ana Oliveira — Belo Horizonte/MG
• Pedro Costa — Curitiba/PR

📍 CLIENTES DE SP:
• João Casimiro — Araraquara
• Maria Silva — São Paulo

👤 BUSCA POR NOME:
• Nome: João Casimiro
• Email: joao@email.com
• Idade: 35

📊 Total de clientes: 5
📊 Clientes de SP: 2


In [5]:
print("✏️ ATUALIZANDO DADOS NO MONGODB")
print("=" * 40)

clientes.update_one(
    {"nome": "Pedro Costa"},
    {"$set": {"cidade": "Florianópolis", "estado": "SC"}}
)
print("✅ Pedro Costa atualizado para Florianópolis/SC!")

produtos.update_one(
    {"nome": "Notebook"},
    {"$set": {"preco": 3200.00},
     "$inc": {"estoque": -1}}
)
print("✅ Notebook — preço atualizado e estoque decrementado!")

print("\n📋 VERIFICANDO ATUALIZAÇÕES:")
pedro = clientes.find_one({"nome": "Pedro Costa"})
print(f"• Pedro: {pedro['cidade']}/{pedro['estado']}")

notebook = produtos.find_one({"nome": "Notebook"})
print(f"• Notebook: R$ {notebook['preco']} — Estoque: {notebook['estoque']}")

✏️ ATUALIZANDO DADOS NO MONGODB
✅ Pedro Costa atualizado para Florianópolis/SC!
✅ Notebook — preço atualizado e estoque decrementado!

📋 VERIFICANDO ATUALIZAÇÕES:
• Pedro: Florianópolis/SC
• Notebook: R$ 3200.0 — Estoque: 9


In [6]:

print("🗑️ DELETANDO DADOS NO MONGODB")
print("=" * 40)


print(f"📊 Clientes antes: {clientes.count_documents({})}")
print(f"📊 Produtos antes: {produtos.count_documents({})}")


clientes.delete_one({"nome": "Pedro Costa"})
print("\n✅ Pedro Costa deletado!")


produtos.delete_one({"nome": "Camiseta"})
print("✅ Camiseta deletada!")


print(f"\n📊 Clientes depois: {clientes.count_documents({})}")
print(f"📊 Produtos depois: {produtos.count_documents({})}")


print("\n📋 CLIENTES RESTANTES:")
for cliente in clientes.find():
    print(f"• {cliente['nome']} — {cliente['cidade']}/{cliente['estado']}")

🗑️ DELETANDO DADOS NO MONGODB
📊 Clientes antes: 5
📊 Produtos antes: 5

✅ Pedro Costa deletado!
✅ Camiseta deletada!

📊 Clientes depois: 4
📊 Produtos depois: 4

📋 CLIENTES RESTANTES:
• João Casimiro — Araraquara/SP
• Maria Silva — São Paulo/SP
• Carlos Santos — Rio de Janeiro/RJ
• Ana Oliveira — Belo Horizonte/MG


In [7]:

print("📊 AGGREGATION PIPELINE")
print("=" * 40)


pedidos_data = [
    {"cliente": "João Casimiro", "produto": "Notebook", "valor": 3200.00, "status": "entregue"},
    {"cliente": "Maria Silva", "produto": "Mouse Gamer", "valor": 150.00, "status": "entregue"},
    {"cliente": "Carlos Santos", "produto": "Tênis Casual", "valor": 280.00, "status": "pendente"},
    {"cliente": "Ana Oliveira", "produto": "Livro Python", "valor": 59.90, "status": "entregue"},
    {"cliente": "João Casimiro", "produto": "Mouse Gamer", "valor": 150.00, "status": "entregue"},
    {"cliente": "Maria Silva", "produto": "Tênis Casual", "valor": 280.00, "status": "cancelado"},
    {"cliente": "Carlos Santos", "produto": "Notebook", "valor": 3200.00, "status": "entregue"}
]

pedidos.insert_many(pedidos_data)
print("✅ Pedidos inseridos!")


print("\n💰 TOTAL GASTO POR CLIENTE:")
pipeline = [
    {"$group": {
        "_id": "$cliente",
        "total_gasto": {"$sum": "$valor"},
        "qtd_pedidos": {"$sum": 1}
    }},
    {"$sort": {"total_gasto": -1}}
]

for doc in pedidos.aggregate(pipeline):
    print(f"• {doc['_id']}: R$ {doc['total_gasto']:.2f} ({doc['qtd_pedidos']} pedidos)")


print("\n📋 PEDIDOS POR STATUS:")
pipeline2 = [
    {"$group": {
        "_id": "$status",
        "quantidade": {"$sum": 1},
        "valor_total": {"$sum": "$valor"}
    }}
]

for doc in pedidos.aggregate(pipeline2):
    print(f"• {doc['_id']}: {doc['quantidade']} pedidos — R$ {doc['valor_total']:.2f}")

📊 AGGREGATION PIPELINE
✅ Pedidos inseridos!

💰 TOTAL GASTO POR CLIENTE:
• Carlos Santos: R$ 3480.00 (2 pedidos)
• João Casimiro: R$ 3350.00 (2 pedidos)
• Maria Silva: R$ 430.00 (2 pedidos)
• Ana Oliveira: R$ 59.90 (1 pedidos)

📋 PEDIDOS POR STATUS:
• cancelado: 1 pedidos — R$ 280.00
• entregue: 5 pedidos — R$ 6759.90
• pendente: 1 pedidos — R$ 280.00


In [8]:

print("🐼 ANÁLISE COM PANDAS")
print("=" * 40)


df_clientes = pd.DataFrame(list(clientes.find()))
df_pedidos = pd.DataFrame(list(pedidos.find()))
df_produtos = pd.DataFrame(list(produtos.find()))


df_clientes = df_clientes.drop('_id', axis=1)
df_pedidos = df_pedidos.drop('_id', axis=1)
df_produtos = df_produtos.drop('_id', axis=1)

print("✅ DataFrames criados!")
print(f"\n📊 Clientes: {df_clientes.shape}")
print(f"📊 Pedidos: {df_pedidos.shape}")
print(f"📊 Produtos: {df_produtos.shape}")

print("\n=== CLIENTES ===")
print(df_clientes)

print("\n=== PRODUTOS ===")
print(df_produtos)

print("\n=== ESTATÍSTICAS DOS PEDIDOS ===")
print(f"Ticket médio: R$ {df_pedidos['valor'].mean():.2f}")
print(f"Maior pedido: R$ {df_pedidos['valor'].max():.2f}")
print(f"Menor pedido: R$ {df_pedidos['valor'].min():.2f}")
print(f"Total em pedidos: R$ {df_pedidos['valor'].sum():.2f}")

🐼 ANÁLISE COM PANDAS
✅ DataFrames criados!

📊 Clientes: (4, 6)
📊 Pedidos: (7, 4)
📊 Produtos: (4, 4)

=== CLIENTES ===
            nome             email  idade          cidade estado  \
0  João Casimiro    joao@email.com     35      Araraquara     SP   
1    Maria Silva   maria@email.com     28       São Paulo     SP   
2  Carlos Santos  carlos@email.com     42  Rio de Janeiro     RJ   
3   Ana Oliveira     ana@email.com     31  Belo Horizonte     MG   

            data_cadastro  
0 2026-09-12 17:00:42.688  
1 2026-09-12 17:00:42.688  
2 2026-09-12 17:00:42.688  
3 2026-09-12 17:00:42.688  

=== PRODUTOS ===
           nome    categoria   preco  estoque
0      Notebook  Eletrônicos  3200.0        9
1   Mouse Gamer  Eletrônicos   150.0       50
2  Tênis Casual     Calçados   280.0       30
3  Livro Python       Livros    59.9       20

=== ESTATÍSTICAS DOS PEDIDOS ===
Ticket médio: R$ 1045.70
Maior pedido: R$ 3200.00
Menor pedido: R$ 59.90
Total em pedidos: R$ 7319.90
